In [168]:
import math

x = [1, 2]
y = [1, 8]

math.sqrt((x[0] - y[0])**2 + (x[1] - y[1])**2)

6.0

In [169]:
(x[0]* y[0] + x[1]*y[1]) / (math.sqrt(1*1+2*2)*math.sqrt(1*1+8*8))

0.9429903335828895

In [170]:
sentence = "this is sample sentence for embedding"
dc = {s:i for i,s in enumerate(sorted(sentence.replace(',', '').split()))}
dc

{'embedding': 0, 'for': 1, 'is': 2, 'sample': 3, 'sentence': 4, 'this': 5}

In [171]:
import torch
vocab_size_tmp = len(dc)
emb = torch.nn.Embedding(vocab_size_tmp, 3)
emb.weight.data 

tensor([[ 1.0699,  0.6228,  0.0282],
        [-1.5694, -0.2350,  0.9434],
        [-2.4308, -0.6562,  0.2244],
        [ 0.1554, -1.0906,  0.5434],
        [-0.3744,  0.8610, -0.6986],
        [ 1.5461,  0.3950, -0.4097]])

In [172]:
sentence_int = torch.tensor( [dc[s] for s in sentence.replace(',', '').split()])
print(sentence_int)

tensor([5, 2, 3, 4, 1, 0])


In [173]:
sentence_emb = emb(sentence_int).detach() # Embedding-Vektoren für den Satz
sentence_emb

tensor([[ 1.5461,  0.3950, -0.4097],
        [-2.4308, -0.6562,  0.2244],
        [ 0.1554, -1.0906,  0.5434],
        [-0.3744,  0.8610, -0.6986],
        [-1.5694, -0.2350,  0.9434],
        [ 1.0699,  0.6228,  0.0282]])

In [174]:
# QKV
d = sentence_emb.shape[1] # Dimensionalität des Embeddings 
d_Q, d_K, d_V = 2, 2, 4

W_Q_L = torch.nn.Linear(d_Q, d, bias=False) # 
# oder 
W_Q = torch.nn.Parameter(torch.rand(d,d_Q)) # trainierbare Parameter 
W_K = torch.nn.Parameter(torch.rand(d,d_K))
W_V = torch.nn.Parameter(torch.rand(d,d_V))

W_Q

Parameter containing:
tensor([[0.7712, 0.3916],
        [0.2056, 0.9445],
        [0.1939, 0.2483]], requires_grad=True)

In [175]:
x_0 = sentence_emb[0] # Erster Token aus Embedding-Satz
q_0 = x_0 @  W_Q
k_0 = x_0 @ W_K
v_0 = x_0 @ W_V 
q_0, k_0, v_0

(tensor([1.1941, 0.8768], grad_fn=<SqueezeBackward4>),
 tensor([0.9169, 1.1960], grad_fn=<SqueezeBackward4>),
 tensor([1.2849, 0.8484, 0.0102, 0.3373], grad_fn=<SqueezeBackward4>))

In [176]:
queries = sentence_emb @ W_Q
keys = sentence_emb @ W_K
values = sentence_emb @ W_V

In [177]:
# Attention 
import torch.nn.functional as F # Funktionen aus Pytorch

scores_raw = queries @ keys.T 
scores = scores_raw / math.sqrt(d_K) # Skalierung
attn_weights = F.softmax(scores, dim=1) # entlang Spalten 
attn_weights # Aufmerksamkeitsgewichte

tensor([[0.4345, 0.0075, 0.0700, 0.0880, 0.0271, 0.3728],
        [0.0009, 0.8677, 0.0197, 0.0135, 0.0970, 0.0012],
        [0.0591, 0.4176, 0.1345, 0.1329, 0.1965, 0.0594],
        [0.2152, 0.1026, 0.1613, 0.1554, 0.1444, 0.2211],
        [0.0226, 0.5658, 0.0971, 0.0792, 0.2095, 0.0259],
        [0.4264, 0.0079, 0.0725, 0.0875, 0.0294, 0.3763]],
       grad_fn=<SoftmaxBackward0>)

In [178]:
attn_output = attn_weights @ values # Gewichtete Summe der Values
attn_output

tensor([[ 1.0390,  0.6045,  0.0661,  0.3832],
        [-2.1901, -1.3504, -0.1363, -0.6849],
        [-1.0574, -0.6812, -0.0360, -0.2946],
        [ 0.1753,  0.0708,  0.0380,  0.1051],
        [-1.5020, -0.9530, -0.0598, -0.4389],
        [ 1.0304,  0.5984,  0.0672,  0.3812]], grad_fn=<MmBackward0>)

In [ ]:
# (1,2,x) -> 1
# (1,2,x) -> 0
# (4,2,x) -> 1
# (5,2,x) -> 0
# (6,2,x) -> 1
# (7,2,x) -> 0

# data leakage

In [179]:
# Masked Attention 

block_size = attn_weights.shape[0]
mask = torch.tril(torch.ones(block_size, block_size), diagonal=-1)
scores_masked = scores.masked_fill(mask.T.bool(), -torch.inf)
attn_weights_masked = torch.softmax(scores_masked, dim=1)
attn_weights_masked

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0010, 0.9990, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0967, 0.6832, 0.2201, 0.0000, 0.0000, 0.0000],
        [0.3391, 0.1616, 0.2543, 0.2450, 0.0000, 0.0000],
        [0.0232, 0.5808, 0.0997, 0.0813, 0.2150, 0.0000],
        [0.4264, 0.0079, 0.0725, 0.0875, 0.0294, 0.3763]],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
# mouse eats _____ <- cheese 

# Ich gehe in die ______ und trinke Bier. <- Kneipe 